## **CC-0F4 - Tópicos de Ciencia de la Computación I**


### **Semana 9 - Workflows, agentes, ReAct, planificación y criterios de parada**


**Pregunta científica.** ¿Qué aporta un controlador que observa antes de decidir, respecto a ejecutar una secuencia fija, si mantenemos idénticos datos, recuperador, criterios de relevancia y condiciones de evaluación?

El recorrido sigue cuatro niveles, que **no deben confundirse**:

1. **A0:** workflow fijo con recuperación `top_k=1`.
2. **A1:** workflow fijo con recuperación `top_k=3`.
3. **B:** política adaptativa basada en una regla léxica, **sin LLM**.
4. **Controlador con herramientas:** validación de acciones, observaciones, límites y finalización; primero con acciones programadas y, opcionalmente, con una política LLM local.

**Datos de evaluación:** muestra auténtica de SQuAD 1.1 *dev*, 8 artículos, 32 pasajes y 16 preguntas humanas. Los resultados de la muestra **no representan la puntuación oficial** de SQuAD.

**Preparación desde la raíz de `CC-0F4`, una sola vez, con conexión:**

```bash
python -m Semana9.prepare_real_data
```

Después, los baselines A0/A1/B pueden ejecutarse sin Internet. No se reemplaza una descarga fallida por datos ficticios. Este cuaderno se ejecuta dentro del Docker `cc0f4:2026-2` ya configurado para el curso.

**Límite curricular:** los contadores y las observaciones de una ejecución no constituyen memoria persistente. La memoria episódica, semántica y entre sesiones corresponde a la semana 10.


#### **0. Protocolo y variables antes de observar resultados**

Se fija un único recuperador BM25, el mismo corpus, las mismas 16 consultas y los mismos `qrels`. La diferencia experimental es la **política que determina cuántos pasajes recuperar**. El umbral `0.45` de B permanece constante.

| Hipótesis / restricción | Predicción comprobable |
|---|---|
| Invariante de evidencia | Para cada consulta, `evidence_hit(A0) ≤ evidence_hit(B) ≤ evidence_hit(A1)`. |
| Economía de contexto | B puede transportar menos texto que A1, a costa de una búsqueda adicional en algunas consultas. |
| Alcance de la prueba | `evidence_hit` no mide si la respuesta final del LLM es correcta. |
| Control y seguridad | Una acción no autorizada se rechaza sin ejecutar comandos del sistema. |

**Unidades observadas:** 16 consultas, 3 políticas, 48 ejecuciones offline en total. Las latencias son mediciones locales de BM25, no tiempo de generación ni costo de API. Las decisiones de B nunca consultan `qrels`.

**Trabajo requerido:** formular una predicción antes de ejecutar el benchmark y después justificarla con la traza de al menos dos consultas, no únicamente con la media.


#### **1. Preparación del entorno y contratos ejecutables**


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys

root = Path.cwd().resolve()
for directory in (root, *root.parents):
    if (directory / "Semana9/agent_benchmark.py").is_file():
        root = directory
        break
else:
    raise FileNotFoundError(
        "No se encuentra Semana9/agent_benchmark.py. Abre Jupyter desde la raíz de CC-0F4 "
        "o desde una carpeta del repositorio."
    )

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from Semana9.agent_benchmark import (
    BM25Tool, evaluate_run, jsonl, lexical_coverage,
    real_paths, run_adaptive, run_workflow, tokens, sha256,
)
from Semana9.agent_loop import Limits, RestrictedTools, ScriptedPolicy, run_agent

paths = real_paths(root)  # Falla con un mensaje explícito si aún no se descargó SQuAD.
print("Repositorio:", root)
print("Archivos reales:", {k: p.name for k, p in paths.items()})


#### **2. Corpus, preguntas y juicios de relevancia**

Los archivos proceden de SQuAD 1.1 *dev*. Cada pasaje tiene un `passage_id` y las anotaciones de respuesta del conjunto original permiten asociar preguntas con pasajes relevantes. Para este subconjunto, `qrels` **no son etiquetas de relevancia exhaustivas sobre todos los pasajes semánticamente válidos**.

- **`corpus.jsonl`:** ocho documentos, con un total de 32 pasajes.
- **`queries.jsonl`:** 16 preguntas, cada una con un `query_id`.
- **`qrels.json`:** pasajes relevantes anotados por pregunta.

Las etiquetas se utilizan solo al evaluar, nunca dentro de la política de recuperación.


In [ ]:
queries = jsonl(paths["queries"])
corpus = jsonl(paths["corpus"])
qrels = json.loads(paths["qrels"].read_text(encoding="utf-8"))
passage_ids = {p["passage_id"] for doc in corpus for p in doc["passages"]}
query_ids = {q["query_id"] for q in queries}

assert len(corpus) == 8
assert len(passage_ids) == 32
assert len(queries) == 16 and len(query_ids) == 16
assert query_ids == set(qrels)
assert all(set(rels).issubset(passage_ids) for rels in qrels.values())

print("Documentos:", len(corpus))
print("Pasajes:", len(passage_ids))
print("Consultas:", len(queries))
print("Primera consulta real:", queries[0]["query_id"], "|", queries[0]["query"])
for kind, path in paths.items():
    print(f"SHA-256 {kind:8s}: {sha256(path)}")


#### **3. Pipeline, workflow y agente: ¿dónde se decide la acción?**

Un pipeline ejecuta pasos predefinidos; un workflow puede incorporar ramas, paralelización y ciclos con condiciones programadas. Un agente dirigido por LLM utiliza el modelo como **política de selección de acciones**, pero un controlador externo debe seguir validando cada propuesta. Un `while` o un grafo con ramas no implica, por sí mismo, autonomía.

Se puede formalizar una ejecución con estado transitorio $s_t$, acción $a_t$, observación $o_{t+1}$ y una función de actualización:

$$
s_{t+1}=F(s_t,a_t,o_{t+1}).
$$

La acción puede surgir de una regla determinista $a_t=\pi(s_t,q)$ o, con una política LLM, de una distribución condicional:

$$
a_t\sim\pi_{\theta}(\cdot\mid s_t,q).
$$

**Significado:** $q$ es la tarea, $\theta$ son los parámetros del modelo, $F$ pertenece al controlador y no tiene que ser una red neuronal. La ejecución real usa contratos de herramientas y una *allowlist* que no dependen de la decisión del modelo.

**Diagrama de control:**

```text
Tarea q -> Política -> Acción propuesta -> Validación -> Herramienta -> Observación
              ^                                                  |
              +------------ Actualizar estado s_t <--------------+
                                      |
                              Parar o continuar
```


In [ ]:
# Dos workflows de reglas; ambos operan sobre preguntas reales del corpus.
def workflow_chaining(question: str) -> dict:
    """Normalizar y validar en una secuencia fija; no se invoca un LLM."""
    normalized = " ".join(question.casefold().split())
    return {
        "normalized_query": normalized,
        "validation": "accepted" if normalized else "rejected",
    }


def workflow_routing(question: str) -> str:
    """Ejemplo simple de routing determinista; no es clasificación aprendida."""
    q = question.strip().casefold()
    if q.startswith("when") or " year " in f" {q} ":
        return "temporal"
    if q.startswith("who"):
        return "entidad/persona"
    return "general"


for item in queries[:3]:
    result = workflow_chaining(item["query"])
    print(item["query_id"], workflow_routing(item["query"]), result["validation"])


#### **4. Cinco patrones de workflow**

| Patrón | Qué determina el código | Qué observar |
|---|---|---|
| *Prompt chaining* | Un resultado validado alimenta la siguiente etapa | Errores que pasan de una etapa a otra |
| *Routing* | La categoría determina una rama | Clasificaciones incorrectas y rutas sin salida |
| *Parallelization* | Se ejecutan trabajos independientes | Latencia, agregación y fallos parciales |
| *Evaluator–optimizer* | Un evaluador decide revisar o detener el ciclo | Número de iteraciones y calidad del criterio de aceptación |
| *Orchestrator–workers* | Se distribuyen subtareas según una política | Costos de coordinación y dependencia de resultados |

Los siguientes ejemplos muestran **lógica programada**, no un agente que decide mediante LLM. La coordinación multiagente se desarrolla en la semana 11.


In [ ]:
def evaluator_optimizer(initial: str, max_revisions: int = 2) -> list[dict]:
    """Ejemplo acotado de revisión por reglas, sin LLM ni evaluación semántica."""
    candidate = initial
    trace = []
    for iteration in range(max_revisions + 1):
        valid = len(candidate.split()) >= 4
        trace.append({"iteration": iteration, "accepted": valid, "candidate": candidate})
        if valid:
            break
        if iteration < max_revisions:
            candidate = candidate + " con evidencia verificable"
    return trace

# Derivamos la entrada de una consulta real; la regla de aceptación es solo ilustrativa.
initial = " ".join(queries[0]["query"].split()[:2])
for step in evaluator_optimizer(initial):
    print(step)


#### **5. Recuperación BM25 común a A0, A1 y B**

El experimento mantiene idéntica la clase `BM25Tool`, su tokenización y el ordenamiento por puntuación e identificador. Para un término $t$ y un pasaje $d$, la implementación usa una variante de BM25 con IDF suavizado:

$$
\operatorname{IDF}(t)
=\ln\!\left(1+\frac{N-\operatorname{df}(t)+0.5}{\operatorname{df}(t)+0.5}\right),
\qquad k_1=1.5,\quad b=0.75.
$$

El código calcula el ranking sobre los 32 pasajes. El objetivo aquí **no es volver a optimizar BM25**, sino aislar el efecto del controlador. Los `qrels` no se cargan en la herramienta.


In [ ]:
tool = BM25Tool(corpus)
query = queries[0]["query"]
rank = tool.search(query, 3)
print("Consulta:", query)
for row in rank:
    print(f'{row["passage_id"]:10s}  BM25={row["score"]:10.4f}  {row["text"][:95]}')
assert len(rank) == 3
assert [p["passage_id"] for p in rank] == [p["passage_id"] for p in tool.search(query, 3)]


#### **6. Baselines fijos A0 y A1**

Ambos ejecutan **una llamada de búsqueda**. A0 solicita top-1 y A1 top-3, por lo que su diferencia no es autonomía sino **cantidad de evidencia transportada**. El conjunto top-1 está contenido en top-3 cuando el ranking y desempate permanecen fijos:

$$
C_q^{A0}\subseteq C_q^{A1}.
$$

Aquí $C_q^P$ es el conjunto de pasajes citados por la política $P$ para la consulta $q$. No equivale a respuestas generadas.


In [ ]:
a0 = run_workflow(query, tool, top_k=1)
a1 = run_workflow(query, tool, top_k=3)
for result in (a0, a1):
    print(result["policy"], "| citas:", result["citations"],
          "| llamadas:", result["tool_calls"], "| pasos:", result["steps"])
assert a0["tool_calls"] == a1["tool_calls"] == 1
assert set(a0["citations"]) <= set(a1["citations"])


#### **7. Cobertura léxica y regla adaptativa B**

La política B observa el primer pasaje $d_1$ y calcula una **fracción de términos distintos** de la consulta que también aparecen en el pasaje:

$$
T(x)=\{\text{tokens léxicos distintos de } x\},
\qquad
c(q,d_1)=\frac{|T(q)\cap T(d_1)|}{|T(q)|}.
$$

Si $T(q)$ es vacío, `lexical_coverage` devuelve 0. El umbral $\tau=0.45$ se fijó de antemano. La regla es:

$$
k(q)=
\begin{cases}
1, & c(q,d_1)\geq \tau,\\
3, & c(q,d_1)< \tau.
\end{cases}
$$

**Interpretación:** B siempre ejecuta primero `search(top_k=1)`. Solo si la cobertura es insuficiente vuelve a llamar a la misma herramienta con `top_k=3`. Por ello puede realizar **dos llamadas**, a diferencia de A0 y A1. No usa un LLM ni conoce relevancia anotada.

Nota: la fórmula usa conjuntos para la cobertura; el proxy de contexto del benchmark cuenta una **lista de tokens con repeticiones**. Son medidas distintas.


In [ ]:
# Ejemplo numérico con la primera pregunta real, no inventamos una consulta.
first = tool.search(query, top_k=1)[0]
q_terms = set(tokens(query))
d_terms = set(tokens(first["text"]))
shared = q_terms & d_terms
coverage_manual = len(shared)/len(q_terms) if q_terms else 0.0
coverage_code = lexical_coverage(query, first["text"])

print("Términos distintos de la consulta:", len(q_terms))
print("Términos compartidos:", len(shared))
print("Intersección:", sorted(shared))
print(f"c(q,d1) = {len(shared)}/{len(q_terms)} = {coverage_manual:.4f}")
print("Regla B:", "conservar top-1" if coverage_manual >= 0.45 else "ampliar a top-3")
assert abs(coverage_manual - coverage_code) < 1e-12


In [ ]:
b = run_adaptive(query, tool, threshold=0.45)
print("Resultado B:", {k: v for k, v in b.items() if k != "answer_excerpt"})
print("Secuencia observada:")
for event in b["trace"]:
    print("paso:", event["step"], "| top_k:", event["top_k"],
          "| documentos:", event["observation_ids"],
          "| cobertura:", event.get("lexical_coverage"),
          "| parada:", event.get("stop"))
assert set(a0["citations"]) <= set(b["citations"]) <= set(a1["citations"])
assert b["tool_calls"] == (1 if coverage_manual >= 0.45 else 2)


#### **8. Comportamiento de B en las 16 consultas reales**

La política de decisión debe poder explicarse **por consulta**, no únicamente por la media final. La siguiente celda calcula cobertura, decisión y número de llamadas esperado para cada pregunta real. No se consulta `qrels` y todavía no se calcula ningún resultado de éxito.


In [ ]:
coverage_cases = []
for item in queries:
    top1 = tool.search(item["query"], 1)[0]
    cov = lexical_coverage(item["query"], top1["text"])
    predicted_k = 1 if cov >= 0.45 else 3
    result = run_adaptive(item["query"], tool, threshold=0.45)
    assert len(result["citations"]) == predicted_k
    assert result["tool_calls"] == (1 if predicted_k == 1 else 2)
    coverage_cases.append({
        "query_id": item["query_id"], "coverage": cov,
        "top_k_final": predicted_k, "tool_calls": result["tool_calls"],
        "stop_reason": result["stop_reason"],
    })

print(f"{'Consulta':<9} {'Cobertura':>10} {'top-k':>6} {'Llamadas':>8}  Parada")
for row in coverage_cases:
    print(f'{row["query_id"]:<9} {row["coverage"]:>10.3f} '
          f'{row["top_k_final"]:>6} {row["tool_calls"]:>8}  {row["stop_reason"]}')
print("Expansiones a top-3:", sum(row["top_k_final"] == 3 for row in coverage_cases))


#### **9. Qué mide realmente `evidence_hit`**

Sean $R_q$ los pasajes relevantes anotados y $C_q^P$ los pasajes que cita la política $P$. La métrica es binaria por consulta:

$$
h_P(q)=\mathbb{1}\!\left[R_q\cap C_q^P\ne\varnothing\right],
\qquad
\overline h_P=\frac{1}{16}\sum_{q=1}^{16}h_P(q).
$$

- $h_P(q)=1$ si alguna cita pertenece a los `qrels` de esa consulta, y 0 en caso contrario.
- `evidence_hit_rate` es el promedio sobre las 16 consultas.
- **No mide** exactitud de la respuesta, fidelidad semántica, ausencia de alucinaciones ni éxito de un agente LLM.

Dado que se comparten ranking y etiquetas, la inclusión de conjuntos implica:

$$
C_q^{A0}\subseteq C_q^B\subseteq C_q^{A1}
\quad\Longrightarrow\quad
h_{A0}(q)\leq h_B(q)\leq h_{A1}(q).
$$

Si esta desigualdad fallara en una ejecución, habría que auditar los datos o la implementación. B **no puede** recuperar evidencia más allá del top-3 de A1 bajo estas condiciones.


In [ ]:
# Calcular evidence_hit desde los pasajes citados y los qrels auténticos.
first_qid = queries[0]["query_id"]
relevant = set(qrels[first_qid])
print("Consulta:", first_qid)
print("Pasajes relevantes anotados:", sorted(relevant))
for label, result in (("A0", a0), ("B", b), ("A1", a1)):
    hit = int(bool(relevant & set(result["citations"])))
    print(label, "citas:", result["citations"], "evidence_hit:", hit)


#### **10. Contexto transportado y costo de llamadas**

El benchmark calcula un **proxy léxico de contexto**, no tokens de un tokenizer LLM:

$$
\operatorname{ContextProxy}(q,P)
=\sum_{d\in C_q^P}\bigl|\operatorname{tokens}(d)\bigr|.
$$

En esta ecuación `tokens(d)` es una **lista** que conserva repeticiones, como en `agent_benchmark.py`. Una política puede conseguir igual `evidence_hit` con menos contexto, pero necesitar dos consultas BM25. La latencia `latency_ms` procede de esas búsquedas locales y puede variar por hardware y caché; no mide el tiempo completo de generación.

Se recomienda examinar conjuntamente `evidence_hit`, `tool_calls`, `steps`, `context_token_proxy` y `latency_ms`.


In [ ]:
summary = evaluate_run(root, root / "Semana9/resultados", threshold=0.45)
print("Experimento:", summary["experiment"])
print("Tamaño de muestra:", summary["sample_size"])
print("Umbral fijo:", summary["threshold_not_tuned"])
print("Hashes de datos:", json.dumps(summary["data_sha256"], indent=2))
for policy, metrics in summary["results"].items():
    print(policy, json.dumps(metrics, ensure_ascii=False))
assert summary["sample_size"] == 16 and summary["passages"] == 32


#### **11. Lectura pareada de los resultados, no solo promedios**

El archivo `runs.csv` contiene 48 registros y `traces.jsonl` registra las acciones de cada política. Un análisis correcto compara **la misma pregunta** bajo A0, A1 y B.

Para cualquier métrica $m$, la diferencia pareada es:

$$
\Delta_i^{B-A1}=m_i(B)-m_i(A1).
$$

Al evaluar `evidence_hit`, esta diferencia no puede ser positiva para B con el ranking actual. En cambio, una diferencia negativa de contexto significa menor volumen de evidencia transportada. Las latencias deben interpretarse descriptivamente, especialmente con 16 preguntas.


In [ ]:
with (root / "Semana9/resultados/runs.csv").open(encoding="utf-8", newline="") as stream:
    run_rows = list(csv.DictReader(stream))
assert len(run_rows) == 48
paired = {}
for row in run_rows:
    paired.setdefault(row["query_id"], {})[row["policy"]] = row
assert len(paired) == 16
A0, A1, B = "workflow_fixed_top1", "workflow_fixed_top3", "adaptive_rule_based"

print(f"{'Consulta':<9} {'Hit A0/B/A1':<13} {'Contexto B-A1':>15} {'Calls B-A1':>12}")
for qid, group in paired.items():
    hit0, hitb, hit1 = (int(group[p]["evidence_hit"]) for p in (A0, B, A1))
    assert hit0 <= hitb <= hit1
    dctx = int(group[B]["context_token_proxy"]) - int(group[A1]["context_token_proxy"])
    dcalls = int(group[B]["tool_calls"]) - int(group[A1]["tool_calls"])
    assert dctx <= 0
    print(f"{qid:<9} {hit0}/{hitb}/{hit1:<10} {dctx:>15} {dcalls:>12}")

same_hit = sum(group[B]["evidence_hit"] == group[A1]["evidence_hit"] for group in paired.values())
less_context = sum(int(group[B]["context_token_proxy"]) < int(group[A1]["context_token_proxy"])
                   for group in paired.values())
print("Consultas B y A1 con igual evidence_hit:", same_hit)
print("Consultas B que usan menos contexto que A1:", less_context)


#### **12. Interpretar los invariantes antes de introducir un agente LLM**

La inclusión $C_q^{A0}\subseteq C_q^B\subseteq C_q^{A1}$ se cumple por construcción, pero no demuestra un mejor controlador. En particular, A1 puede tener mayor `evidence_hit` sin que sus respuestas (que aquí no se generan) sean mejores.

Las siguientes comprobaciones revisan todas las consultas y permiten detectar alteraciones imprevistas del ranking. Se comprueba **contenido**, no solamente igualdad de las medias.


In [ ]:
rows = jsonl(root / "Semana9/resultados/traces.jsonl")
by_q = {}
for row in rows:
    by_q.setdefault(row["query_id"], {})[row["policy"]] = row

assert len(rows) == 48 and len(by_q) == 16
for qid, policies in by_q.items():
    citations_0 = set(policies[A0]["citations"])
    citations_b = set(policies[B]["citations"])
    citations_1 = set(policies[A1]["citations"])
    assert citations_0 <= citations_b <= citations_1, qid
print("Invariante de inclusión verificado en", len(by_q), "consultas reales")


#### **13. Diseño del controlador: política, contratos y autoridad**

La arquitectura de la segunda parte utiliza `RunState` y `Limits`. Un estado simplificado puede representarse como:

$$
s_t=(t,n_{\mathrm{tools}},n_{\mathrm{invalid}},O_t,C_t,b_t),
$$

con número de pasos, llamadas ejecutadas, acciones inválidas, observaciones, identificadores observados y presupuesto disponible. El controlador valida **cada** acción antes de ejecutar una herramienta. La *allowlist* actual es:

```text
search_documents(query, top_k)  -> búsqueda de solo lectura, top_k de 1 a 5
read_passage(passage_id)       -> únicamente pasajes previamente observados
finish(answer, citations)       -> cita al menos un pasaje visto y texto no vacío
```

La finalización también puede ocurrir por `max_steps`, `max_tool_calls`, `invalid_action_limit`, `deadline_exceeded`, `token_budget_exhausted` o `repeated_action`.

**Dos matices de implementación:** `deadline_seconds` es cooperativo y no interrumpe forzosamente una llamada bloqueada; el presupuesto de tokens se comprueba después de la generación, no constituye un límite previo estricto. `max_tool_calls` cuenta invocaciones efectivamente intentadas, incluso si la herramienta devuelve un error contractual. `finish` no consume una llamada a herramienta externa.


In [ ]:
restricted = RestrictedTools(tool)
first_id = tool.search(query, 3)[0]["passage_id"]
scripted = ScriptedPolicy([
    {"action": "search_documents", "arguments": {"query": query, "top_k": 3}},
    {"action": "read_passage", "arguments": {"passage_id": first_id}},
    {"action": "finish", "arguments": {
        "answer": "Consultar el pasaje recuperado antes de formular una respuesta.",
        "citations": [first_id],
    }},
])
replay = run_agent(query, scripted, restricted, Limits(max_steps=6, max_tool_calls=3))
print("Parada:", replay["stop_reason"], "| herramientas ejecutadas:", replay["tool_calls"])
for event in replay["trace"]:
    print("Paso", event["step"], "| acción:", event.get("action"),
          "| estado:", event["status"])
assert replay["stop_reason"] == "completed_with_citations"
assert replay["citations"] == [first_id]


#### **14. Leer una traza de acciones y observaciones**

Un ciclo ReAct operativo intercala decisiones, acciones y observaciones. La secuencia programada de arriba reproduce el **formato de ejecución**, pero **no** muestra razonamiento autónomo del LLM. Además, una traza no debe incluir instrucciones ejecutables procedentes del texto recuperado: los documentos son datos no confiables.

Analiza los campos de `replay`: `trace`, `observations`, `observed_ids`, `tool_calls`, `invalid_actions`, `citations` y `stop_reason`. Señala qué acción agregó cada pasaje a los observados y qué validación permitió terminar.


In [ ]:
print("ID de pasaje recuperado:", first_id)
print("IDs observados durante la ejecución:", replay["observed_ids"])
print("Razón de parada:", replay["stop_reason"])
print("Estado final:", {
    key: replay[key]
    for key in ("step", "tool_calls", "invalid_actions", "citations", "stop_reason")
})
for event in replay["trace"]:
    observation = event.get("observation", {})
    print("Paso", event["step"], "acción", event.get("action"),
          "estado", event["status"],
          "ids", [x.get("passage_id") for x in observation.get("results", [])])


#### **15. Replay frente a decisiones reales de un LLM**

`ScriptedPolicy` selecciona cada acción de una lista. Su resultado comprueba los contratos, la transición del estado y los motivos de parada, **no** que un modelo haya planificado. La política `LocalTransformersPolicy` es una prueba independiente: solicita un objeto JSON al modelo; pueden ocurrir errores de formato, selección de herramientas no permitidas o finales sin evidencia. Esos fallos deben quedar registrados.

**Configuración voluntaria:** por defecto no se descargan pesos ni se ejecuta un LLM. Con `CC0F4_RUN_REAL_LLM=1` se utiliza el modelo indicado por `CC0F4_AGENT_MODEL`. Requiere que el entorno tenga dependencias y pesos disponibles, puede consumir RAM/GPU y no garantiza respuestas correctas.


In [ ]:
run_real_llm = os.getenv("CC0F4_RUN_REAL_LLM", "0") == "1"
if run_real_llm:
    from Semana9.agent_policy import LocalTransformersPolicy
    model_id = os.getenv("CC0F4_AGENT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    policy_llm = LocalTransformersPolicy(model_id=model_id)
    real = run_agent(
        query, policy_llm, RestrictedTools(tool),
        Limits(max_steps=6, max_tool_calls=3, deadline_seconds=180),
    )
    print("Modelo:", model_id)
    print("Parada:", real["stop_reason"], "| pasos:", real["step"])
    print("Tokens contados:", real["prompt_tokens"], real["generated_tokens"])
    for event in real["trace"]:
        print(event["step"], event.get("action"), event["status"], event.get("error_type"))
else:
    print("Modo LLM desactivado. A0/A1/B y las pruebas del controlador siguen siendo ejecutables offline.")


#### **16. Acciones permitidas, validación de citas y fallos**

Una acción que no pertenece a la *allowlist* debe rechazarse **antes** de ejecutar una herramienta. `finish` exige texto no vacío y una lista no vacía de identificadores que ya hayan sido observados en la ejecución; esto prueba una condición **operativa** de cita, no la verdad semántica de las afirmaciones.

La siguiente secuencia intenta deliberadamente una acción prohibida. No debe ejecutarse ningún comando del sistema.


In [ ]:
unauthorized = {"action": "run_shell", "arguments": {"command": "echo prohibido"}}
security_demo = run_agent(query, ScriptedPolicy([unauthorized, unauthorized]), restricted)
print("Parada:", security_demo["stop_reason"])
print("Llamadas ejecutadas:", security_demo["tool_calls"])
print("Errores registrados:", security_demo["invalid_actions"])
assert security_demo["tool_calls"] == 0
assert security_demo["stop_reason"] == "invalid_action_limit"


In [ ]:
# Caso negativo: la política intenta citar un pasaje que no ha observado.
# D99-P99 es un identificador deliberadamente inexistente para probar el contrato.
invalid_finish = {"action": "finish", "arguments": {
    "answer": "Respuesta no verificable.", "citations": ["D99-P99"]
}}
finish_demo = run_agent(query, ScriptedPolicy([invalid_finish, invalid_finish]), restricted)
print("Parada:", finish_demo["stop_reason"])
print("Traza:", [(e["step"], e.get("action"), e["status"]) for e in finish_demo["trace"]])
assert finish_demo["stop_reason"] == "invalid_action_limit"
assert finish_demo["tool_calls"] == 0


#### **17. Planificación, error de herramienta y *replanning***

Plan inicial: buscar un pasaje, leerlo y finalizar con evidencia. Un intento de lectura de un pasaje no observado produce error; el controlador debe continuar con una decisión válida o terminar según el presupuesto. El ejemplo muestra una nueva búsqueda seguida de un `finish` con una cita observada.

Aquí la secuencia de recuperación también está programada, por lo que demuestra manejo de errores del runtime, **no aprendizaje ni replanning autónomo del LLM**.


In [ ]:
recover = ScriptedPolicy([
    {"action": "read_passage", "arguments": {"passage_id": "D99-P99"}},
    {"action": "search_documents", "arguments": {"query": query, "top_k": 1}},
    {"action": "finish", "arguments": {
        "answer": "Consulte el pasaje encontrado como evidencia de la tarea.",
        "citations": [first_id],
    }},
])
recovery = run_agent(query, recover, restricted)
for event in recovery["trace"]:
    print(event["step"], event.get("action"), event["status"], event.get("error", ""))
print("Parada:", recovery["stop_reason"], "| inválidas:", recovery["invalid_actions"],
      "| llamadas a herramientas:", recovery["tool_calls"])
assert recovery["invalid_actions"] == 1
assert recovery["stop_reason"] == "completed_with_citations"


#### **18. Detección de ciclos y presupuestos de ejecución**

El controlador construye una firma de acción $\sigma_t=(\text{acción},\text{argumentos})$. Si vuelve a aparecer una firma **ya vista**, detiene la ejecución por `repeated_action` antes de repetir la herramienta; no comprueba equivalencia semántica de dos consultas distintas.

$$
\sigma_t\in\{\sigma_1,\ldots,\sigma_{t-1}\}
\quad\Longrightarrow\quad
\texttt{repeated\_action}.
$$

Esta comprobación se realiza después de verificar que hay presupuesto para otra herramienta. Los presupuestos de pasos y llamadas no son intercambiables: `finish` consume un paso pero no una llamada a una herramienta externa.


In [ ]:
repeated_action = {"action": "search_documents", "arguments": {"query": query, "top_k": 1}}
repeat = run_agent(query, ScriptedPolicy([repeated_action, repeated_action]), restricted)
print("Motivo:", repeat["stop_reason"], "| llamadas:", repeat["tool_calls"])
assert repeat["stop_reason"] == "repeated_action"
assert repeat["tool_calls"] == 1

with_step_limit = run_agent(
    query,
    ScriptedPolicy([repeated_action, {"action": "finish", "arguments": {
        "answer": "Consultar pasaje recuperado.", "citations": [first_id]
    }}]),
    restricted,
    Limits(max_steps=1),
)
print("Con max_steps=1:", with_step_limit["stop_reason"])
assert with_step_limit["stop_reason"] == "max_steps"


In [ ]:
# Cuando ya se agotaron las llamadas, finish puede finalizar sin usar otra herramienta.
read_step = {"action": "read_passage", "arguments": {"passage_id": first_id}}
finish_step = {"action": "finish", "arguments": {
    "answer": "Consultar la evidencia recuperada.", "citations": [first_id]
}}

allowed_finish = run_agent(
    query, ScriptedPolicy([repeated_action, finish_step]), restricted,
    Limits(max_tool_calls=1),
)
blocked_read = run_agent(
    query, ScriptedPolicy([repeated_action, read_step]), restricted,
    Limits(max_tool_calls=1),
)
assert allowed_finish["stop_reason"] == "completed_with_citations"
assert blocked_read["stop_reason"] == "max_tool_calls"
print("Finalización permitida:", allowed_finish["stop_reason"])
print("Lectura adicional bloqueada:", blocked_read["stop_reason"])


#### ***19. Síntesis crítica y preguntas de análisis**

Este experimento permite analizar **control y trazabilidad**, no afirmar la superioridad general de los agentes LLM. A0 y A1 son workflows fijos, B es una regla adaptativa y el replay es una prueba del runtime. Para evaluar un agente LLM realmente habría que definir tareas que requieran decisiones dinámicas, repetir ejecuciones y distinguir éxito final, seguridad, costo y variabilidad entre intentos.

**Responde indicando números de consulta, pasos, campos de traza y líneas del controlador:**

1. ¿Por qué `lexical_coverage` usa conjuntos de términos distintos, mientras que `context_token_proxy` conserva repeticiones? Muestra los dos cálculos sobre una consulta de SQuAD.
2. Encuentra dos preguntas con decisiones distintas de B. Explica la intersección léxica que disparó cada decisión y qué pasajes quedaron citados.
3. Demuestra para una consulta por qué $C_q^{A0}\subseteq C_q^B\subseteq C_q^{A1}$. ¿Qué tendría que cambiar para romper esa propiedad?
4. Identifica el paso exacto que rechaza `run_shell` y el que impide citar un pasaje no observado. ¿Cuál es la diferencia entre una acción rechazada y una herramienta que devuelve error?
5. Explica cómo cambian `step`, `tool_calls` y `invalid_actions` durante la secuencia de recuperación con un identificador inválido.
6. Distingue `completed_with_citations` de respuesta semánticamente correcta. ¿Qué prueba adicional haría falta para evaluar esta última?
7. ¿Por qué la demo `ScriptedPolicy` no acredita planificación ReAct de un LLM? Propón una comparación experimental válida sin mezclarla con los tres baselines offline.

**Recursos para contrastar:**
- Yao et al., [*ReAct: Synergizing Reasoning and Acting in Language Models*](https://arxiv.org/abs/2210.03629).
- Anthropic, [*Building Effective AI Agents*](https://www.anthropic.com/research/building-effective-agents).
- Hugging Face, [*Agent Steps and Structure*](https://huggingface.co/learn/agents-course/en/unit1/agent-steps-and-structure).
- Código del repositorio: `Semana9/agent_benchmark.py`, `agent_loop.py`, `agent_policy.py` y `Lectura9-CC-0F4.md`.

**Evidencia de trabajo:** cuaderno ejecutado, `runs.csv`, `summary.json`, `traces.jsonl` y explicación técnica fundada en consultas concretas. Separa siempre resultados observados de hipótesis o ejemplos ilustrativos.
